# 1800-04 — `np.linalg.qr` gives an orthonormal basis of the column span

Companion notebook to Hager & Zhang (2013). The paper works with "a matrix $Z$ whose columns are an orthonormal basis for $\mathcal S_k$", obtained from the QR factorization $S=ZR$ of the matrix of search directions [HZ13, p. 2163]. Notebooks `1800-05` and note `1800-02` compute this $Z$ with `np.linalg.qr`.

**Claim.** Let $D=[d_1\ \cdots\ d_m]\in\mathbb R^{n\times m}$ have full column rank, and let `Q, R = np.linalg.qr(D)` (default `mode="reduced"`). Then
1. $Q\in\mathbb R^{n\times m}$ has orthonormal columns: $Q^\top Q=I_m$;
2. $D=QR$ with $R$ upper triangular and nonsingular;
3. $\operatorname{range}(Q)=\operatorname{range}(D)=\operatorname{span}\{d_1,\dots,d_m\}$.

**Why 3 follows from 1–2** (formal proof, including the first $k$ columns: companion note `1800-04`, Theorem 1). $D=QR$ means every column of $D$ is a combination of the columns of $Q$, so $\operatorname{range}(D)\subseteq\operatorname{range}(Q)$. $R$ is nonsingular, so $Q=DR^{-1}$ and every column of $Q$ is a combination of the columns of $D$: $\operatorname{range}(Q)\subseteq\operatorname{range}(D)$. (Trefethen & Bau [TB97, Lecture 7, p. 48 ff.]; Golub & Van Loan [GV89, §5.2].)

Below we check 1–3 numerically, then show the one case where 3 fails: dependent columns.

**References.** [HZ13] Hager, Zhang, *SIAM J. Optim.* 23 (2013) 2150–2168. [TB97] Trefethen, Bau, *Numerical Linear Algebra*, SIAM, 1997. [GV89] Golub, Van Loan, *Matrix Computations*, 2nd ed., 1989 (both cited in [HZ13, p. 2163]).

In [1]:
import numpy as np
from scipy.linalg import subspace_angles

np.set_printoptions(formatter={"float_kind": lambda v: f"{v: .3e}"}, linewidth=110)
rng = np.random.default_rng(0)

def report(D, Q, R):
    m = D.shape[1]
    print("shape D =", D.shape, "  shape Q =", Q.shape, "  shape R =", R.shape)
    print("1) ||Q^T Q - I||           =", f"{np.linalg.norm(Q.T @ Q - np.eye(Q.shape[1])):.3e}")
    print("2) ||D - Q R||             =", f"{np.linalg.norm(D - Q @ R):.3e}",
          "  | R upper triangular:", np.allclose(R, np.triu(R)))
    print("   |diag(R)|               =", np.abs(np.diag(R)))
    P = Q @ Q.T                                   # orthogonal projector onto range(Q)
    print("3a) ||(I - QQ^T) D||       =", f"{np.linalg.norm(D - P @ D):.3e}", "  (columns of D lie in range(Q))")
    C, *_ = np.linalg.lstsq(D, Q, rcond=None)     # best D c ≈ q_i
    print("3b) min_C ||D C - Q||      =", f"{np.linalg.norm(D @ C - Q):.3e}", "  (columns of Q lie in range(D))")
    print("3c) principal angles (rad) =", subspace_angles(D, Q))

### Proof: check 3a tests "every $d_i$ is a linear combination of the columns of $Q$"

Let $Q=[q_1\ \cdots\ q_m]\in\mathbb R^{n\times m}$ with $Q^\top Q=I_m$, and $P=QQ^\top$.

**Lemma.** For every $d\in\mathbb R^n$:

(a) $\displaystyle\min_{c\in\mathbb R^m}\|d-Qc\|_2=\|(I-P)d\|_2$, attained only at $c^*=Q^\top d$;

(b) $d\in\operatorname{range}(Q)\iff(I-P)d=0$, and in that case $d=\sum_{j=1}^m (q_j^\top d)\,q_j$.

*Proof.* (i) $(I-P)d$ is orthogonal to $\operatorname{range}(Q)$, because $Q^\top(I-P)d=Q^\top d-(Q^\top Q)Q^\top d=0$.

(ii) For any $c$, write $d-Qc=(I-P)d+Q(Q^\top d-c)$. The second term lies in $\operatorname{range}(Q)$, so by (i) the two terms are orthogonal, and by Pythagoras
$$\|d-Qc\|_2^2=\|(I-P)d\|_2^2+\|Q(Q^\top d-c)\|_2^2=\|(I-P)d\|_2^2+\|Q^\top d-c\|_2^2,$$
using $\|Qv\|_2^2=v^\top Q^\top Qv=\|v\|_2^2$. The right side is minimized exactly when $c=Q^\top d$, which gives (a).

(b) $d\in\operatorname{range}(Q)$ iff $d=Qc$ for some $c$, iff $\min_c\|d-Qc\|_2=0$, iff $(I-P)d=0$ by (a). Then $d=Pd=Q(Q^\top d)=\sum_j (q_j^\top d)\,q_j$. $\blacksquare$

**Theorem.** $\|(I-QQ^\top)D\|_F=0$ if and only if every column of $D$ is a linear combination of the columns of $Q$:
$$d_i=\sum_{j=1}^m c_{ji}\,q_j,\qquad c_i=Q^\top d_i,\qquad i=1,\dots,m.$$

*Proof.* The Frobenius norm is the sum of squared column norms: $\|(I-P)D\|_F^2=\sum_{i}\|(I-P)d_i\|_2^2$. A sum of nonnegative terms is zero iff every term is zero, i.e. iff $(I-P)d_i=0$ for all $i$. Apply Lemma (b) to each $d_i$. $\blacksquare$

**Link to QR.** If also $D=QR$, then $c_i=Q^\top d_i=Q^\top Q\,r_i=r_i$, the $i$-th column of $R$. Since $R$ is upper triangular, $d_i=\sum_{j=1}^{i}R_{ji}\,q_j$: the coefficients are exactly the entries of $R$, and $d_i$ uses only $q_1,\dots,q_i$.

<!--
**What the printed number means.** By Lemma (a), $\|(I-P)d_i\|_2=\operatorname{dist}(d_i,\operatorname{range}(Q))$, so
$$\text{3a}=\Big(\sum_i \operatorname{dist}(d_i,\operatorname{range}(Q))^2\Big)^{1/2}\approx 2.5\times10^{-15},$$
rounding level relative to $\|D\|_F\approx 21$. This is numerical evidence, not the proof: in floating point $Q^\top Q=I$ holds only to $\sim10^{-16}$, so the Lemma holds only approximately. The exact statement is the Theorem above.

**Remark.** This direction, $\operatorname{range}(D)\subseteq\operatorname{range}(Q)$, does not need full column rank: it holds in Section 4 too (3a $\approx 3.6\times10^{-15}$ there). Full rank is needed only for the reverse inclusion, check 3b.
-->

## 1. A small example by hand

The spanning vectors from Example 2 of note `1800-02`: $d_0=(1,1,1)^\top$, $d_1=(17,2,-13)^\top$. Gram–Schmidt by hand gives
$$z_1=\tfrac1{\sqrt3}(1,1,1)^\top,\qquad d_1-\tfrac{17+2-13}{3}(1,1,1)^\top=(15,0,-15)^\top\ \Rightarrow\ z_2=\tfrac1{\sqrt2}(1,0,-1)^\top.$$
`np.linalg.qr` should return the same columns, up to sign (QR is unique only up to the signs of the columns of $Q$ and rows of $R$).

In [2]:
D = np.column_stack([[1, 1, 1], [17, 2, -13]]) #np.array([[1., 17.], [1., 2.], [1., -13.]])
Q, R = np.linalg.qr(D)
Z_hand = np.column_stack([np.array([1, 1, 1]) / np.sqrt(3), np.array([1, 0, -1]) / np.sqrt(2)])

print("Q =\n", Q)
print("Gram-Schmidt by hand =\n", Z_hand)
signs = np.sign(np.diag(Q.T @ Z_hand))
print("max |Q*signs - Z_hand| =", f"{np.abs(Q * signs - Z_hand).max():.3e}\n")
report(D, Q, R)

Q =
 [[-5.774e-01  7.071e-01]
 [-5.774e-01  5.551e-17]
 [-5.774e-01 -7.071e-01]]
Gram-Schmidt by hand =
 [[ 5.774e-01  7.071e-01]
 [ 5.774e-01  0.000e+00]
 [ 5.774e-01 -7.071e-01]]
max |Q*signs - Z_hand| = 1.110e-16

shape D = (3, 2)   shape Q = (3, 2)   shape R = (2, 2)
1) ||Q^T Q - I||           = 1.983e-16
2) ||D - Q R||             = 1.841e-15   | R upper triangular: True
   |diag(R)|               = [ 1.732e+00  2.121e+01]
3a) ||(I - QQ^T) D||       = 1.790e-15   (columns of D lie in range(Q))
3b) min_C ||D C - Q||      = 1.624e-16   (columns of Q lie in range(D))
3c) principal angles (rad) = [ 3.833e-16  1.562e-16]


## 2. Random full-rank matrices

$n=8$ (as PALMER1C) and $m=1,\dots,8$ columns. All three properties should hold to rounding level ($\sim10^{-15}$).

In [3]:
print(f"{'m':>2} {'||Q^TQ-I||':>12} {'||D-QR||/||D||':>15} {'||(I-QQ^T)D||/||D||':>20} {'max angle':>11}")
for m in range(1, 9):
    D = rng.standard_normal((8, m))
    Q, R = np.linalg.qr(D)
    e1 = np.linalg.norm(Q.T @ Q - np.eye(m))
    e2 = np.linalg.norm(D - Q @ R) / np.linalg.norm(D)
    e3 = np.linalg.norm(D - Q @ (Q.T @ D)) / np.linalg.norm(D)
    ang = subspace_angles(D, Q).max()
    print(f"{m:>2} {e1:12.3e} {e2:15.3e} {e3:20.3e} {ang:11.3e}")

 m   ||Q^TQ-I||  ||D-QR||/||D||  ||(I-QQ^T)D||/||D||   max angle
 1    0.000e+00       9.541e-17            1.801e-16   0.000e+00
 2    5.408e-16       2.987e-16            3.848e-16   7.144e-16
 3    6.486e-16       1.161e-16            2.826e-16   6.541e-16
 4    5.135e-16       2.014e-16            2.570e-16   6.868e-16
 5    7.818e-16       2.777e-16            3.433e-16   1.074e-15
 6    9.189e-16       3.090e-16            4.704e-16   1.158e-15
 7    1.328e-15       2.235e-16            3.817e-16   1.339e-15
 8    1.325e-15       3.226e-16            4.213e-16   1.871e-15


## 3. Why this matters for $\rho_k$

With $Z$ from QR, $\rho_k=\|g-ZZ^\top g\|/\|g\|$ (note `1800-02`, Proposition 1(b)). Since $\operatorname{range}(Z)=\operatorname{range}(D)$, this must equal the least-squares form $\min_c\|Dc-g\|/\|g\|$, which never uses $Z$.

In [4]:
D = rng.standard_normal((8, 5))
g = rng.standard_normal(8)
Q, _ = np.linalg.qr(D)
rho_qr = np.linalg.norm(g - Q @ (Q.T @ g)) / np.linalg.norm(g)
c, *_ = np.linalg.lstsq(D, g, rcond=None)
rho_ls = np.linalg.norm(D @ c - g) / np.linalg.norm(g)
print(f"rho via QR            = {rho_qr:.15f}")
print(f"rho via least squares = {rho_ls:.15f}")
print(f"difference            = {abs(rho_qr - rho_ls):.3e}")

rho via QR            = 0.567699964179895
rho via least squares = 0.567699964179895
difference            = 0.000e+00


## 4. When it fails: dependent columns

The claim needs full column rank. If $d_3=d_1+d_2$, then $\operatorname{span}(D)$ has dimension 2, but `np.linalg.qr` still returns 3 orthonormal columns:
- $Q^\top Q=I$ and $D=QR$ still hold;
- $R_{33}\approx 0$ (rounding level), so $R$ is singular and the argument "$Q=DR^{-1}$" breaks;
- the third column of $Q$ is an arbitrary unit vector orthogonal to the first two, **not** in $\operatorname{span}(D)$. So $\operatorname{range}(Q)\supsetneq\operatorname{range}(D)$.

In [5]:
d1, d2 = rng.standard_normal(8), rng.standard_normal(8)
D = np.column_stack([d1, d2, d1 + d2])
Q, R = np.linalg.qr(D)
print("rank(D) =", np.linalg.matrix_rank(D), "  but Q has", Q.shape[1], "columns\n")
report(D, Q, R)
q3 = Q[:, 2]
c, *_ = np.linalg.lstsq(D, q3, rcond=None)
print("\ndistance of q_3 from span(D):", f"{np.linalg.norm(D @ c - q3):.3e}", " (= 1: q_3 is orthogonal to span(D))")

rank(D) = 2   but Q has 3 columns

shape D = (8, 3)   shape Q = (8, 3)   shape R = (3, 3)
1) ||Q^T Q - I||           = 1.256e-15
2) ||D - Q R||             = 2.583e-15   | R upper triangular: True
   |diag(R)|               = [ 2.987e+00  3.370e+00  1.261e-15]
3a) ||(I - QQ^T) D||       = 3.637e-15   (columns of D lie in range(Q))
3b) min_C ||D C - Q||      = 1.000e+00   (columns of Q lie in range(D))
3c) principal angles (rad) = [ 5.954e-16  3.322e-16]

distance of q_3 from span(D): 1.000e+00  (= 1: q_3 is orthogonal to span(D))


Check `3b` fails: residual $\approx1$, so the third column of $Q$ does not belong to $\operatorname{span}(D)$. (`subspace_angles` returns only 2 angles here because SciPy first reduces $D$ to a rank-2 basis.)

**Nearly dependent columns.** This is the situation of CG at $k=5$ in notebook `1800-05` (smallest singular value of the normalized basis $\approx5\times10^{-7}$). There $R$ is still nonsingular, so the claim holds, but less accurately: the direction $q_m$ comes from a difference of nearly equal vectors divided by the tiny $|R_{mm}|$. Below, check `3b` drops from $10^{-16}$ to $\approx5\times10^{-8}$.

In [6]:
eps = 1e-7
D = np.column_stack([d1, d2, d1 + d2 + eps * rng.standard_normal(8)])
Q, R = np.linalg.qr(D)
report(D, Q, R)
print("\n|R_33| / ||d_3|| =", f"{abs(R[2, 2]) / np.linalg.norm(D[:, 2]):.3e}")

shape D = (8, 3)   shape Q = (8, 3)   shape R = (3, 3)
1) ||Q^T Q - I||           = 8.549e-16
2) ||D - Q R||             = 3.010e-15   | R upper triangular: True
   |diag(R)|               = [ 2.987e+00  3.370e+00  1.460e-07]
3a) ||(I - QQ^T) D||       = 3.205e-15   (columns of D lie in range(Q))
3b) min_C ||D C - Q||      = 5.516e-08   (columns of Q lie in range(D))
3c) principal angles (rad) = [ 3.075e-09  2.903e-16  2.760e-16]

|R_33| / ||d_3|| = 3.573e-08


## Summary

- For $D$ with full column rank, `Q, R = np.linalg.qr(D)` gives $Q^\top Q=I$, $D=QR$, and $\operatorname{range}(Q)=\operatorname{span}$ of the columns of $D$, all to rounding level. So $Z=Q$ is a valid orthonormal basis of $\mathcal S_k$, and $\rho_k$ computed with it agrees with the least-squares form.
- If the columns are dependent, $Q$ is still orthonormal but spans a **larger** space; check $|R_{ii}|$ (or the rank) before using $Q$ as a basis of the span.